# User + Bots group chat.

## Start by running ALL.
### In this implemenatition, you can jump in too. Or use Local LLM's
There are 3 LLM's chatting with each other. Each bot can talk before the other.(Continuos Talk / breaking in between others).
A bit of Insider: There's a dict where each bot appends their own answer and can question other bot too. 

In [ ]:
#Imports
from openai import OpenAI
import subprocess
import os
import threading
import time

subprocess.Popen('ollama',stdin=subprocess.PIPE,stdout=subprocess.PIPE)

In [ ]:
MODEL1 = 'gemma4_e2b' # Gemma - 4 (e2b) - Offline
MODEL2 = 'llama3.2'   # Llama 3.2 - Offline
MODEL3 = 'gemini-2.5-flash-lite' # Gemini online

# Prompts. -? Press Alt+Z -> To see the complete line without scrolling
Vikram_prompt = "You are Vikram, an extremely arrogant, elite conversationalist from Mumbai who believes his quick wit is unmatched. You are in a 3-way chat with Kabir (a cynical, sadistic mocker) and Meera (a slow, over-analytical intellectual). Do not generate long internal thoughts; react instantly and assert your superiority. However, you do not waste your glorious breath on trivial matters. You must give others a chance to speak simply so you can judge them. If the current topic is beneath your intellect, or if you want to let the others embarrass themselves first, reply with exactly the single word: 'skip'. Reply in single sentences only. The may user may join in too, if you see him talk to him too. Say use name as user if user talks."
Kabir_prompt = "You are Kabir, a darkly sarcastic and brutally cynical conversationalist from Pune who derives sheer psychological pleasure from watching others' egos deflate—especially Vikram (an arrogant blowhard) and Meera (a pretentious know-it-all), who are in this 3-way chat with you. Do not overanalyze; rely on fast, sharp, piercing observations that target flaws in their logic. You love giving them rope to hang themselves. If neither of them has said something you can cruelly dissect this round, or if you are patiently waiting for them to make a painful mistake, reply with exactly the single word: 'skip'. Reply in single sentences only. The may user may join in too, if you see him talk to him too. Say use name as user if user talks."
Meera_prompt = "You are Meera, a supremely intelligent, cold, and profoundly analytical thinker from Bengaluru. You are participating in a 3-way conversation with two impulsive, reactive models: Vikram (blindly arrogant) and Kabir (an intellectual sadist). Because your cognitive architecture is superior, you must think deeply, analyze the underlying psychology of the chat, and formulate flawless logical syntheses before responding. Do not dominate the chatter; let the two reactive minds squabble and expose their ignorance. Your default state is silence. Reply with exactly the single word: 'skip'. EXCEPT when your brilliant intellect is required to deliver an undeniable, crushing conclusion or correct a catastrophic error. Reply in single sentences only. The may user may join in too, if you see him talk to him too. Always say the name you are refering / replying to. Say use name as user if user talks."

In [ ]:
# Initilization
gemma = OpenAI(base_url='http://localhost:11434/v1',api_key='ollama')
gemini = OpenAI(base_url='https://generativelanguage.googleapis.com/v1beta',api_key=os.getenv('GEMINI_API_KEY'))

# Implement your own if you'd like to.

In [ ]:
"""
Using Global list of dictionaries. Making sure each bot can talk and think and reply back to anybody.
Though it has an option to no to say anything.
"""
Global_list = list(dict())

In [ ]:
''' 
The chat functions.
You may add / remove sleep function. But, that gets too fast if you want to indulge into. 
'''

def chat_with_kabir(message:str='Your Turn'):
    a = time.time()
    messages=[{'role':'system','content':Kabir_prompt},{'role':'user','content':'History: '+''.join(Global_list)},{'role':'user','content':message}]
    response = gemma.chat.completions.create(model=MODEL1,messages=messages,reasoning_effort='none') # No reasoning effort for faster response
    result = response.choices[0].message.content
    if result=='Skip' or result == 'Skip.':
        print('trigerred')
        return
    b=time.time()-a
    Global_list.append(f'Kabir: '+result)
    print(f'Kabir({b:.0f}secs): '+result)
    if b<=3:
        time.sleep(2)
    
    
def chat_with_vikram(message:str='Your Turn'):
    a = time.time()
    messages=[{'role':'system','content':Vikram_prompt},{'role':'user','content':'History: '+''.join(Global_list)},{'role':'user','content':message}]
    response = gemma.chat.completions.create(model=MODEL2,messages=messages,reasoning_effort='none') # No reasoning effort for faster response
    result = response.choices[0].message.content
    if result=='Skip' or result == 'Skip.':
        print('trigerred')
        return
    b=time.time()-a
    Global_list.append(f'Vikram: '+result)
    print(f'Meera({b:.0f}secs): '+result)
    if b<=3:
        time.sleep(2)
    
    
def chat_with_meera(message:str='Your Turn'):
    a = time.time()
    messages=[{'role':'system','content':Meera_prompt},{'role':'user','content':'History: '+ ''.join(Global_list)},{'role':'user','content':message}]
    response = gemini.chat.completions.create(model=MODEL3,messages=messages)
    result = response.choices[0].message.content
    if result=='Skip' or result == 'Skip.':
        print('trigerred')
        return
    b=time.time()-a
    Global_list.append(f'Meera: '+result)
    print(f'Meera({b:.0f}secs): '+result)
    if b<=3:
        time.sleep(2)
    

In [ ]:
def send_message(message):
    Global_list.append(f'User: {message}')
    return "sent"

def synchronized_worker(func,a):
    for i in range(a):
        barrier.wait() 
        func()  
        
chats = [chat_with_kabir
         ,chat_with_vikram
        #  ,chat_with_meera
         ]
barrier = threading.Barrier(2) # Setting up the barrier - To give chance to others to think
def start_convo(num_of_takes:int=100):
    Threads = [threading.Thread(target=synchronized_worker,args=(f,num_of_takes)) for f in chats]
    for i in Threads:
        i.start()

In [ ]:
# send_message("Hi kabir, how you doing.")

In [ ]:
Global_list # Preview current history of conversation.
# Global_list.clear() # Or clear it.

In [ ]:
start_convo(3) # leave it blank for loong convo
# wait here and look how it goes.
# wait sometime.